In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install matplotlib_scalebar

In [3]:
import sys, os

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/MR MCD Pipeline/2D"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print("Added to sys.path:", PROJECT_ROOT)

print("Project root content:", os.listdir(PROJECT_ROOT))


Added to sys.path: /content/drive/MyDrive/Colab Notebooks/MR MCD Pipeline/2D
Project root content: ['Notebooks', 'data', 'Magnetisation']


In [4]:
import numpy as np
import json
import torch
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar

import sys
import os
sys.path.append(os.getcwd()) # Add current directory to Python path

from Magnetisation.Propagator import Propagator
from Magnetisation.Generator import generator_CNN
from Magnetisation.Train import Magnetisation_CNN_training
from Magnetisation.utils import LoadData

In [5]:
import numpy as np
# 兼容补丁 / Compatibility patch
if not hasattr(np, "int"):
  np.int = int

In [6]:
!pip install discretisedfield micromagneticmodel oommfc


In [7]:
import discretisedfield as df
import micromagneticmodel as mm
import oommfc as oc
import numpy as np
import matplotlib.pyplot as plt


In [8]:
Lx = 1e-6
Ly = 1e-6
Lz = 5e-9

cell = (5e-9, 5e-9, 5e-9)

mesh = df.Mesh(
    p1=(-Lx/2, -Ly/2, 0),
    p2=( Lx/2,  Ly/2, Lz),
    cell=cell
)

Ms = 8e5
A = 1e-11

system = mm.System(name="uniform_OOP")
system.energy = mm.Exchange(A=A) + mm.Demag()


In [9]:
def uniform(pos):
    return (0, 0, Ms)  # OOP only

system.m = df.Field(mesh, nvdim=3, value=uniform)

md = oc.MinDriver()
md.drive(system)

m = system.m.sel(z=Lz/2)

Mx = np.squeeze(m.x.array)
My = np.squeeze(m.y.array)
Mz = np.squeeze(m.z.array)


OSError: Cannot find OOMMF.

In [ ]:
nx, ny = Mz.shape
dx = cell[0]
dy = cell[1]

kx = 2*np.pi*np.fft.fftfreq(nx, d=dx)
ky = 2*np.pi*np.fft.fftfreq(ny, d=dy)
kx, ky = np.meshgrid(kx, ky, indexing="ij")

k = np.sqrt(kx**2 + ky**2)
k[0,0] = 1e-12

Mz_k = np.fft.fft2(Mz)

mu0 = 4*np.pi*1e-7

# Thin-film OOP kernel (same as Recon_OOP_mag)
Bz_k = mu0 * (k * Mz_k) / (2*k)

# Propagate to NV height
z_NV = 50e-9
Bz_k *= np.exp(-k*z_NV)

Bz = np.real(np.fft.ifft2(Bz_k))


In [ ]:
plt.figure(figsize=(5,4))
plt.imshow(Bz*1e3, origin="lower", cmap="seismic")
plt.colorbar(label="Bz (mT)")
plt.title("Forward Bz at 50 nm")
plt.show()


In [ ]:
Bz_meas_k = np.fft.fft2(Bz)

# Remove propagation
Bz_surface_k = Bz_meas_k * np.exp(k*z_NV)

# Inversion kernel (same model)
Mz_recon_k = (2 * Bz_surface_k) / (mu0 * k)
Mz_recon_k[0,0] = 0

Mz_recon = np.real(np.fft.ifft2(Mz_recon_k))


In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(Mz, origin="lower", cmap="viridis")
plt.title("Ground Truth Mz")
plt.colorbar()

plt.subplot(1,3,2)
plt.imshow(Mz_recon, origin="lower", cmap="viridis")
plt.title("Reconstructed Mz")
plt.colorbar()

plt.subplot(1,3,3)
plt.imshow(Mz_recon - Mz, origin="lower", cmap="seismic")
plt.title("Difference")
plt.colorbar()

plt.tight_layout()
plt.show()

print("Max reconstruction error:", np.max(np.abs(Mz_recon - Mz)))
